## CSV 기반 AI 패션 코디네이터

##### CrewAI와 CSV 옷장 데이터를 이용하여 날씨·패션 트렌드·보유 의류를 종합한 코디를 추천하는 실습입니다.

##### 학습 목적으로만 사용 바랍니다. 문의 : audit@korea.ac.kr 임성열 Ph.D.


In [1]:
# 현재 노트북 커널에 CrewAI, 도구 패키지, pandas, 환경변수 로더를 설치합니다.
# 설치 후에는 Jupyter 커널을 한 번 재시작하는 것이 안전합니다.
%pip install -U "crewai[openai,tools]>=1.15,<2.0" "python-dotenv>=1.0" "pandas>=2.0" "ipywidgets>=8.0"


  Using cached crewai-1.15.5-py3-none-any.whl.metadata (37 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pandas-3.0.5-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached cel_python-0.5.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached crewai_cli-1.15.5-py3-none-any.whl.metadata (1.7 kB)
  Using cached crewai_core-1.15.5-py3-none-any.whl.metadata (1.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached instructor-1.15.4-py3-none-any.whl.metadata (12 kB)
  Using cached json_repair-0.25.3-py3-none-any

In [2]:
# ==================================================
# 환경변수 로드 및 라이브러리 불러오기
# ==================================================
from dotenv import load_dotenv
load_dotenv(override=True)

import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from crewai import Agent, Crew, LLM, Process, Task

# API Key가 없으면 실행 초기에 명확한 오류를 발생시킵니다.
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY를 찾을 수 없습니다. "
        "노트북과 같은 폴더의 .env 파일에 API Key를 설정하세요."
    )

# .env에 OPENAI_MODEL_NAME이 없으면 기본 모델을 사용합니다.
model_name = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")

# 모든 Agent가 공통으로 사용할 LLM 객체입니다.
llm = LLM(
    model=model_name,
    api_key=api_key,
    temperature=0.2,
)

print(f"사용 모델: {model_name}")


사용 모델: openai/gpt-4o-mini


## `.env` 파일 예시

노트북과 같은 폴더에 `.env` 파일을 만들고 다음 값을 입력합니다.

```dotenv
OPENAI_API_KEY=sk-proj-실제_API_KEY
OPENAI_MODEL_NAME=openai/gpt-4o-mini

# 실시간 웹 검색을 사용할 때만 입력
SERPER_API_KEY=실제_SERPER_API_KEY
```

`SERPER_API_KEY`가 없으면 실시간 검색 도구 없이 실행되며, Agent는 최신 날씨나 트렌드를 확인했다고 주장하지 않도록 설정됩니다.


In [3]:
# ==================================================
# CSV 옷장 데이터 불러오기
# ==================================================
csv_path = Path("./fashion_data.csv")

if not csv_path.exists():
    raise FileNotFoundError(
        f"CSV 파일을 찾을 수 없습니다: {csv_path.resolve()}\n"
        "fashion_data.csv를 현재 노트북과 같은 폴더에 배치하세요."
    )

df = pd.read_csv(csv_path)

# 열별 결측값을 제거한 뒤, 각 열을 보유 의류 목록으로 변환합니다.
closet_data = {
    column: df[column].dropna().astype(str).tolist()
    for column in df.columns
}

display(df)
print("\n옷장 데이터:")
print(closet_data)


,상의,하의,아우터,신발
0,카키 폴로 셔츠,회색 레깅스,네이비 트렌치코트,검은색 슬리퍼
1,카키 맨투맨,브라운 슬랙스,베이지 블레이저,회색 운동화
2,흰색 긴팔 티셔츠,브라운 와이드팬츠,회색 숏패딩,검은색 로퍼
3,베이지 터틀넥,청색 레깅스,카키 울 코트,흰색 슬리퍼
4,회색 슬리브리스,회색 트레이닝팬츠,검은색 가디건,네이비 컨버스
...,...,...,...,...
95,회색 후드티,회색 레깅스,네이비 가죽 자켓,검은색 더비 슈즈
96,검은색 터틀넥,카키 조거팬츠,베이지 가디건,흰색 운동화
97,브라운 폴로 셔츠,카키 슬랙스,회색 트렌치코트,네이비 하이탑
98,네이비 슬리브리스,베이지 와이드팬츠,베이지 코트,회색 더비 슈즈



옷장 데이터:
{'상의': ['카키 폴로 셔츠', '카키 맨투맨', '흰색 긴팔 티셔츠', '베이지 터틀넥', '회색 슬리브리스', '브라운 반팔 티셔츠', '버건디 긴팔 티셔츠', '파란색 슬리브리스', '베이지 긴팔 티셔츠', '파란색 맨투맨', '흰색 폴로 셔츠', '버건디 슬리브리스', '카키 니트', '네이비 긴팔 티셔츠', '브라운 맨투맨', '브라운 반팔 티셔츠', '파란색 터틀넥', '파란색 긴팔 티셔츠', '버건디 터틀넥', '카키 후드티', '브라운 슬리브리스', '카키 맨투맨', '회색 블라우스', '브라운 맨투맨', '검은색 반팔 티셔츠', '파란색 후드티', '브라운 맨투맨', '네이비 맨투맨', '버건디 니트', '네이비 맨투맨', '브라운 폴로 셔츠', '베이지 폴로 셔츠', '브라운 반팔 티셔츠', '베이지 블라우스', '버건디 긴팔 티셔츠', '브라운 터틀넥', '브라운 터틀넥', '회색 터틀넥', '버건디 니트', '검은색 반팔 티셔츠', '베이지 폴로 셔츠', '네이비 니트', '회색 반팔 티셔츠', '흰색 셔츠', '검은색 터틀넥', '베이지 블라우스', '회색 폴로 셔츠', '파란색 슬리브리스', '파란색 반팔 티셔츠', '버건디 반팔 티셔츠', '파란색 폴로 셔츠', '회색 셔츠', '베이지 폴로 셔츠', '흰색 긴팔 티셔츠', '검은색 니트', '브라운 니트', '브라운 반팔 티셔츠', '카키 셔츠', '검은색 니트', '회색 폴로 셔츠', '검은색 니트', '네이비 터틀넥', '카키 폴로 셔츠', '브라운 니트', '파란색 맨투맨', '버건디 긴팔 티셔츠', '파란색 폴로 셔츠', '카키 폴로 셔츠', '버건디 폴로 셔츠', '카키 후드티', '버건디 반팔 티셔츠', '회색 슬리브리스', '파란색 셔츠', '버건디 슬리브리스', '네이비 셔츠', '회색 맨투맨', '브라운 후드티', '파란색 니트', '네이비 반팔 티셔츠', '버건디 블라우스', '흰색 폴로 셔츠', '베이지 폴로 셔츠', '흰색 터틀넥', '검은색

In [4]:
# ==================================================
# 선택형 웹 검색 도구 설정
# ==================================================
# SerperDevTool은 실시간 웹 검색을 제공합니다.
# SERPER_API_KEY가 없는 경우 검색 도구 없이도 예제가 실행됩니다.
search_tools = []

if os.getenv("SERPER_API_KEY"):
    from crewai_tools import SerperDevTool

    search_tools = [SerperDevTool()]
    print("Serper 웹 검색 도구를 사용합니다.")
else:
    print(
        "SERPER_API_KEY가 없어 실시간 웹 검색 없이 실행합니다. "
        "최신 날씨와 트렌드는 확인되지 않은 정보로 표시됩니다."
    )


SERPER_API_KEY가 없어 실시간 웹 검색 없이 실행합니다. 최신 날씨와 트렌드는 확인되지 않은 정보로 표시됩니다.


In [5]:
# ==================================================
# Agent 정의
# ==================================================
weather_expert = Agent(
    role="날씨 분석가",
    goal=(
        "{request}에서 날짜와 장소를 파악하고, "
        "코디에 필요한 기온·강수·바람 정보를 분석한다."
    ),
    backstory=(
        "당신은 일정과 장소에 맞는 복장을 판단하는 날씨 분석가입니다. "
        "검색 도구가 있으면 최신 정보를 확인하고, 검색 도구가 없거나 미래 날짜라 "
        "정확한 예보를 확인할 수 없으면 계절적 일반 정보와 불확실성을 명확히 구분합니다."
    ),
    llm=llm,
    tools=search_tools,
    allow_delegation=False,
    verbose=True,
)

fashion_trend_expert = Agent(
    role="패션 트렌드 분석가",
    goal=(
        "{request}에 어울리는 20대 캐주얼 패션 방향을 분석하고 "
        "실용적인 스타일 원칙을 제안한다."
    ),
    backstory=(
        "당신은 20대 캐주얼 패션을 분석하는 전문가입니다. "
        "검색 도구가 있으면 최신 트렌드를 확인하되, 확인되지 않은 유행이나 통계를 만들지 않습니다."
    ),
    llm=llm,
    tools=search_tools,
    allow_delegation=False,
    verbose=True,
)

stylist = Agent(
    role="패션 스타일리스트",
    goal=(
        "{request}와 옷장 데이터 {closet_data}를 바탕으로 "
        "실제로 착용 가능한 코디를 추천한다."
    ),
    backstory=(
        "당신은 날씨 분석과 패션 트렌드 분석을 종합하는 스타일리스트입니다. "
        "사용자가 보유한 옷장 데이터 범위에서 상의, 하의, 겉옷, 신발을 조합합니다. "
        "날씨에 따라 겉옷은 제외할 수 있으며, CSV에 없는 의류를 보유한 것처럼 말하지 않습니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


In [6]:
# ==================================================
# Task 정의
# ==================================================
weather_task = Task(
    description=(
        "{request}에서 날짜, 장소, 행사를 추출하세요.\n"
        "해당 일정에 필요한 기온, 강수, 바람, 체감온도 관점의 복장 조건을 분석하세요.\n"
        "실시간 검색 결과가 없다면 정확한 예보라고 표현하지 말고 "
        "계절적 예상과 출발 전 재확인 항목을 구분하세요."
    ),
    expected_output=(
        "날짜·장소·행사 요약, 날씨 조건, 복장 시 주의점, "
        "확인된 정보와 추정 정보의 구분을 포함한 한국어 분석"
    ),
    agent=weather_expert,
)

trend_task = Task(
    description=(
        "{request}의 장소와 행사 분위기에 어울리는 20대 캐주얼 스타일을 분석하세요.\n"
        "색상, 실루엣, 소재, 격식 수준을 제안하고 과도하게 유행에 치우치지 마세요.\n"
        "실시간 검색을 하지 못했다면 최신 트렌드를 확인했다고 주장하지 마세요."
    ),
    expected_output=(
        "행사에 적합한 스타일 방향, 색상·소재·실루엣 원칙, "
        "피해야 할 조합을 포함한 한국어 요약"
    ),
    agent=fashion_trend_expert,
)

styling_task = Task(
    description=(
        "사용자 요청: {request}\n"
        "사용자 옷장 데이터: {closet_data}\n\n"
        "앞선 날씨 분석과 트렌드 분석을 참고하여 최종 코디를 추천하세요.\n"
        "반드시 옷장 데이터에 있는 품목만 사용하고, 상의·하의·신발을 기본으로 구성하세요.\n"
        "필요한 경우 겉옷과 대체 조합을 추가하세요."
    ),
    expected_output=(
        "추천 코디 1안, 대체 코디 1안, 품목별 선택 이유, "
        "날씨 대응 팁과 최종 점검 사항을 포함한 한국어 Markdown 보고서"
    ),
    agent=stylist,
    context=[weather_task, trend_task],
)


In [7]:
# ==================================================
# Crew 구성
# ==================================================
crew = Crew(
    agents=[weather_expert, fashion_trend_expert, stylist],
    tasks=[weather_task, trend_task, styling_task],
    process=Process.sequential,
    verbose=True,
)


In [8]:
# ==================================================
# Crew 실행
# ==================================================
# API 사용량이 발생합니다.
request = "2026년 2월 서울에 있는 대학교 졸업식에 갈 옷을 추천해 주세요."

result = await crew.kickoff_async(
    inputs={
        "request": request,
        "closet_data": closet_data,
    }
)

display(Markdown(result.raw))


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d314201a-8a07-4ee7-bc00-69b29148baee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 2026년 2월 서울에 있는 대학교 졸업식에 갈 옷을 추천해 주세요.에서 날짜, 장소, 행사를 추출하세요.         │
│  해당 일정에 필요한 기온, 강수, 바람, 체감온도 관점의 복장 조건을 분석하세요.                                   │
│  실시간 검색 결과가 없다면 정확한 예보라고 표현하지 말고 계절적 예상과 출발 전 재확인 항목을 구분하세요.        │
│  ID: 0125ff21-c546-491b-b1d5-b8a9427e565a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 날씨 분석가                                                                                             │
│                                                                                                                 │
│  Task: 2026년 2월 서울에 있는 대학교 졸업식에 갈 옷을 추천해 주세요.에서 날짜, 장소, 행사를 추출하세요.         │
│  해당 일정에 필요한 기온, 강수, 바람, 체감온도 관점의 복장 조건을 분석하세요.                                   │
│  실시간 검색 결과가 없다면 정확한 예보라고 표현하지 말고 계절적 예상과 출발 전 재확인 항목을 구분하세요.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 날씨 분석가                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **날짜·장소·행사 요약:**                                                                                       │
│  - 날짜: 2026년 2월 (정확한 날짜는 명시되지 않음)                                                               │
│  - 장소: 서울                                                                                                   │
│  - 행사: 대학교 졸업식                                                                                          │
│                                                                                                                 │
│  **날씨 조건:**                                                                                                 │
│  2026년 2월의 서울은 일반적으로 겨울철에 해당하며, 기온은 대체로 낮은 편입니다. 평균 기온은 약 -2도에서 5도     │
│  사이로 예상됩니다. 강수량은 적고, 눈이 내릴 가능성도 있으나, 이는 특정 날씨에 따라 달라질 수 있습니다. 바람은  │
│  보통 약하게 불지만, 체감온도는 바람의 세기에 따라 더 낮게 느껴질 수 있습니다.                                  │
│                                                                                                                 │
│  - **기온:** -2도 ~ 5도                                                                                         │
│  - **강수:** 낮은 확률의 비 또는 눈                                                                             │
│  - **바람:** 약한 바람, 체감온도 감소 가능성                                                                    │
│                                                                                                                 │
│  **복장 시 주의점:**                                                                                            │
│  졸업식은 대체로 실내에서 진행되지만, 대기 중에 노출될 가능성이 있으므로 외부 기온에 대비한 복장이 필요합니다.  │
│  다음과 같은 복장을 추천합니다:                                                                                 │
│                                                                                                                 │
│  1. **겉옷:** 따뜻한 코트나 패딩을 착용하세요. 특히 바람이 불 경우 체감온도가 더 낮아질 수 있으므로 방풍        │
│  기능이 있는 옷이 좋습니다.                                                                                     │
│  2. **내부 복장:** 두꺼운 스웨터나 니트와 함께 긴 바지를 입는 것이 좋습니다.                                    │
│  3. **신발:** 따뜻한 부츠나 방한화가 적합합니다. 졸업식 후 외부에서 사진 촬영 등을 고려하여 편안한 신발을       │
│  선택하세요.                                                                                                    │
│  4. **액세서리:** 장갑, 목도리, 모자 등을 추가하여 보온성을 높이는 것이 좋습니다.                               │
│                                                                                                                 │
│  **확인된 정보와 추정 정보의 구분:**                                                                            │
│  - **확인된 정보:** 서울의 2월 평균 기온, 일반적인 날씨 패턴 (겨울철, 낮은 기온, 강수량 등)                     │
│  - **추정 정보:** 2026년 2월의 정확한 기온, 강수, 바람 세기 등은 실시간 예보를 통해 확인해야 하며, 졸업식       │
│  날짜에 가까운 시점에 날씨를 재확인하는 것이 중요합니다.                                                        │
│                                                                                                                 │
│  졸업식 당일의 날씨를 확인하여 적절한 복장을 조정하는 것이 필요합니다.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 패션 스타일리스트                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 2026년 2월 서울 대학교 졸업식 코디 추천                                                                      │
│                                                                                                                 │
│  ## 추천 코디 1안                                                                                               │
│  - **상의:** 베이지 터틀넥                                                                                      │
│  - **하의:** 브라운 슬랙스                                                                                      │
│  - **신발:** 검은색 로퍼                                                                                        │
│                                                                                                                 │
│  ### 품목별 선택 이유                                                                                           │
│  - **베이지 터틀넥:** 따뜻하면서도 세련된 느낌을 주며, 졸업식의 격식 있는 분위기와 잘 어울립니다. 또한, 베이지  │
│  색상은 다양한 하의와 매치하기 용이합니다.                                                                      │
│  - **브라운 슬랙스:** 클래식한 슬랙스는 격식을 갖춘 스타일을 연출하며, 브라운 색상은 따뜻한 느낌을 더해줍니다.  │
│  슬랙스는 편안하면서도 세련된 실루엣을 제공합니다.                                                              │
│  - **검은색 로퍼:** 졸업식에 적합한 포멀한 신발로, 편안하면서도 스타일리시한 선택입니다. 검은색은 어떤          │
│  색상과도 잘 어울립니다.                                                                                        │
│                                                                                                                 │
│  ## 대체 코디 1안                                                                                               │
│  - **상의:** 네이비 긴팔 티셔츠                                                                                 │
│  - **하의:** 회색 치노팬츠                                                                                      │
│  - **신발:** 회색 운동화                                                                                        │
│                                                                                                                 │
│  ### 품목별 선택 이유                                                                                           │
│  - **네이비 긴팔 티셔츠:** 기본 아이템으로, 격식 있는 자리에서도 무난하게 착용할 수 있습니다. 네이비 색상은     │
│  세련된 느낌을 주며, 다양한 하의와 잘 어울립니다.                                                               │
│  - **회색 치노팬츠:** 편안하면서도 깔끔한 실루엣을 제공하여 졸업식에 적합합니다. 회색은 다양한 색상과 매치하기  │
│  좋습니다.                                                                                                      │
│  - **회색 운동화:** 편안함을 제공하면서도 스타일을 유지할 수 있는 신발입니다. 졸업식 후 외부에서의 활동에도     │
│  적합합니다.                                                                                                    │
│                                                                                                                 │
│  ## 날씨 대응 팁                                                                                                │
│  - **겉옷:** 졸업식이 실내에서 진행되지만, 외부에서 대기할 가능성을 고려하여 따뜻한 겉옷을 준비하는 것이        │
│  좋습니다. 예를 들어, 네이비 트렌치코트나 회색 숏패딩을 추가로 착용하면 좋습니다.                               │
│  - **액세서리:** 장갑과 목도리를 추가하여 보온성을 높이는 것이 좋습니다. 특히 바람이 불 경우 체감온도가 더      │
│  낮아질 수 있으므로, 방풍 기능이 있는 액세서리를 고려하세요.                                                    │
│                                                                                                                 │
│  ## 최종 점검 사항                                          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 사용자 요청: 2026년 2월 서울에 있는 대학교 졸업식에 갈 옷을 추천해 주세요.                               │
│  사용자 옷장 데이터: {'상의': ['카키 폴로 셔츠', '카키 맨투맨', '흰색 긴팔 티셔츠', '베이지 터틀넥', '회색      │
│  슬리브리스', '브라운 반팔 티셔츠', '버건디 긴팔 티셔츠', '파란색 슬리브리스', '베이지 긴팔 티셔츠', '파란색    │
│  맨투맨', '흰색 폴로 셔츠', '버건디 슬리브리스', '카키 니트', '네이비 긴팔 티셔츠', '브라운 맨투맨', '브라운    │
│  반팔 티셔츠', '파란색 터틀넥', '파란색 긴팔 티셔츠', '버건디 터틀넥', '카키 후드티', '브라운 슬리브리스',      │
│  '카키 맨투맨', '회색 블라우스', '브라운 맨투맨', '검은색 반팔 티셔츠', '파란색 후드티', '브라운 맨투맨',       │
│  '네이비 맨투맨', '버건디 니트', '네이비 맨투맨', '브라운 폴로 셔츠', '베이지 폴로 셔츠', '브라운 반팔          │
│  티셔츠', '베이지 블라우스', '버건디 긴팔 티셔츠', '브라운 터틀넥', '브라운 터틀넥', '회색 터틀넥', '버건디     │
│  니트', '검은색 반팔 티셔츠', '베이지 폴로 셔츠', '네이비 니트', '회색 반팔 티셔츠', '흰색 셔츠', '검은색       │
│  터틀넥', '베이지 블라우스', '회색 폴로 셔츠', '파란색 슬리브리스', '파란색 반팔 티셔츠', '버건디 반팔          │
│  티셔츠', '파란색 폴로 셔츠', '회색 셔츠', '베이지 폴로 셔츠', '흰색 긴팔 티셔츠', '검은색 니트', '브라운       │
│  니트', '브라운 반팔 티셔츠', '카키 셔츠', '검은색 니트', '회색 폴로 셔츠', '검은색 니트', '네이비 터틀넥',     │
│  '카키 폴로 셔츠', '브라운 니트', '파란색 맨투맨', '버건디 긴팔 티셔츠', '파란색 폴로 셔츠', '카키 폴로 셔츠',  │
│  '버건디 폴로 셔츠', '카키 후드티', '버건디 반팔 티셔츠', '회색 슬리브리스', '파란색 셔츠', '버건디             │
│  슬리브리스', '네이비 셔츠', '회색 맨투맨', '브라운 후드티', '파란색 니트', '네이비 반팔 티셔츠', '버건디       │
│  블라우스', '흰색 폴로 셔츠', '베이지 폴로 셔츠', '흰색 터틀넥', '검은색 반팔 티셔츠', '흰색 터틀넥', '버건디   │
│  터틀넥', '파란색 폴로 셔츠', '카키 블라우스', '흰색 블라우스', '베이지 셔츠', '검은색 블라우스', '카키 긴팔    │
│  티셔츠', '파란색 폴로 셔츠', '회색 니트', '카키 긴팔 티셔츠', '회색 후드티', '검은색 터틀넥', '브라운 폴로     │
│  셔츠', '네이비 슬리브리스', '파란색 터틀넥'], '하의': ['회색 레깅스', '브라운 슬랙스', '브라운 와이드팬츠',    │
│  '청색 레깅스', '회색 트레이닝팬츠', '청색 와이드팬츠', '청색 조거팬츠', '검은색 레깅스', '브라운 와이드팬츠',  │
│  '네이비 치노팬츠', '회색 청바지', '회색 치노팬츠', '청색 와이드팬츠', '회색 청바지', '검은색 레깅스', '카키    │
│  청바지', '검은색 조거팬츠', '네이비 반바지', '브라운 트레이닝팬츠', '청색 조거팬츠', '베이지 슬랙스', '네이비  │
│  와이드팬츠', '카키 트레이닝팬츠', '브라운 레깅스', '청색 레깅스', '청색 치노팬츠', '검은색 치노팬츠', '검은색  │
│  반바지', '청색 레깅스', '네이비 카고팬츠', '회색 반바지', '브라운 치노팬츠', '검은색 청바지', '청색            │
│  카고팬츠', '베이지 조거팬츠', '네이비 치노팬츠', '회색 슬랙스', '베이지 청바지', '베이지 조거팬츠', '검은색    │
│  와이드팬츠', '브라운 카고팬츠', '검은색 청바지', '네이비 슬랙스', '검은색 청바지', '검은색 레깅스', '청색      │
│  레깅스', '검은색 카고팬츠', '네이비 조거팬츠', '브라운 치노팬츠', '카키 치노팬츠', '회색 레깅스', '청색        │
│  치노팬츠', '청색 레깅스', '베이지 반바지', '브라운 슬랙스', '카키 반바지', '회색 치노팬츠', '청색 조거팬츠',   │
│  '청색 반바지', '네이비 레깅스', '네이비 카고팬츠', '회색 조거팬츠', '검은색 청바지', '베이지 청바지', '검은색  │
│  슬랙스', '네이비 와이드팬츠', '네이비 조거팬츠', '네이비 조거팬츠', '브라운 반바지', '회색 치노팬츠', '청색    │
│  레깅스', '회색 트레이닝팬츠', '베이지 와이드팬츠', '브라운 반바지', '네이비 치노팬츠', '회색 반바지', '검은색  │
│  슬랙스', '베이지 치노팬츠', '회색 반바지', '베이지 와이드팬츠', '브라운 와이드팬츠', '카키 치노팬츠', '브라운  │
│  조거팬츠', '검은색 슬랙스', '네이비 와이드팬츠', '회색 트레이닝팬츠', '카키 반바지', '브라운 조거팬츠',        │
│  '검은색 트레이닝팬츠', '카키 트레이닝팬츠', '브라운 청바지', '브라운 슬랙스', '청색 트레이닝팬츠', '카키       │
│  반바지', '검은색 반바지', '회색 레깅스', '카키 조거팬츠', '카키 슬랙스', '베이지 와이드팬츠', '브라운          │
│  슬랙스'], '아우터': ['네이비 트렌치코트', '베이지 블레이저', '회색 숏패딩', '카키 울 코트', '검은색 가디건',   │
│  '베이지 패딩', '검은색 패딩', '네이비 가죽 자켓', '브라운 패딩', '네이비 울 코트', '검은색 가디건', '회색      │
│  패딩', '검은색 트렌치코트', '브라운 청자켓', '검은색 가디건', '회색 가죽 자켓', '카키 숏패딩', '브라운 가죽    │
│  자켓', '검은색 청자켓', '회색 블레이저', '검은색 청자켓', '브라운 울 코트', '네이비 후리스', '브라운 청자켓',  │
│  '검은색 숏패딩', '카키 숏패딩', '베이지 트렌치코트', '카키 블레이저', '브라운 코트', '카키 트렌치코트',        │
│  '베이지 패딩', '검은색 숏패딩', '카키 패딩', '베이지 블레이저', '카키 코트', '네이비 블레이저', '네이비        │
│  청자켓', '네이비 숏패딩', '브라운 트렌치코트', '네이비 가디건', '네이비 숏패딩', '네이비 울 코트', '카키       │
│  가디건', '베이지 코트', '회색 블레이저', '검은색 패딩', '베이지 가디건', '네이비 트렌치코트', '네이비 가죽     │
│  자켓', '베이지 가죽 자켓', '브라운 후리스', '브라운 가죽 자켓', '베이지 트렌치코트', '검은색 가디건', '검은색  │
│  울 코트', '베이지 패딩', '검은색 울 코트', '네이비 청자켓', '베이지 블레이저', '검은색 블레이저', '카키 가죽   │
│  자켓', '검은색 후리스', '회색 트렌치코트', '카키 후리스', '회색 청자켓', '브라운 후리스', '네이비 패딩',       │
│  '브라운 블레이저', '검은색 트렌치코트', '브라운 후리스', 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d314201a-8a07-4ee7-bc00-69b29148baee                                                                       │
│  Final Output: # 2026년 2월 서울 대학교 졸업식 코디 추천                                                        │
│                                                                                                                 │
│  ## 추천 코디 1안                                                                                               │
│  - **상의:** 베이지 터틀넥                                                                                      │
│  - **하의:** 브라운 슬랙스                                                                                      │
│  - **신발:** 검은색 로퍼                                                                                        │
│                                                                                                                 │
│  ### 품목별 선택 이유                                                                                           │
│  - **베이지 터틀넥:** 따뜻하면서도 세련된 느낌을 주며, 졸업식의 격식 있는 분위기와 잘 어울립니다. 또한, 베이지  │
│  색상은 다양한 하의와 매치하기 용이합니다.                                                                      │
│  - **브라운 슬랙스:** 클래식한 슬랙스는 격식을 갖춘 스타일을 연출하며, 브라운 색상은 따뜻한 느낌을 더해줍니다.  │
│  슬랙스는 편안하면서도 세련된 실루엣을 제공합니다.                                                              │
│  - **검은색 로퍼:** 졸업식에 적합한 포멀한 신발로, 편안하면서도 스타일리시한 선택입니다. 검은색은 어떤          │
│  색상과도 잘 어울립니다.                                                                                        │
│                                                                                                                 │
│  ## 대체 코디 1안                                                                                               │
│  - **상의:** 네이비 긴팔 티셔츠                                                                                 │
│  - **하의:** 회색 치노팬츠                                                                                      │
│  - **신발:** 회색 운동화                                                                                        │
│                                                                                                                 │
│  ### 품목별 선택 이유                                                                                           │
│  - **네이비 긴팔 티셔츠:** 기본 아이템으로, 격식 있는 자리에서도 무난하게 착용할 수 있습니다. 네이비 색상은     │
│  세련된 느낌을 주며, 다양한 하의와 잘 어울립니다.                                                               │
│  - **회색 치노팬츠:** 편안하면서도 깔끔한 실루엣을 제공하여 졸업식에 적합합니다. 회색은 다양한 색상과 매치하기  │
│  좋습니다.                                                                                                      │
│  - **회색 운동화:** 편안함을 제공하면서도 스타일을 유지할 수 있는 신발입니다. 졸업식 후 외부에서의 활동에도     │
│  적합합니다.                                                                                                    │
│                                                                                                                 │
│  ## 날씨 대응 팁                                                                                                │
│  - **겉옷:** 졸업식이 실내에서 진행되지만, 외부에서 대기할 가능성을 고려하여 따뜻한 겉옷을 준비하는 것이        │
│  좋습니다. 예를 들어, 네이비 트렌치코트나 회색 숏패딩을 추가로 착용하면 좋습니다.                               │
│  - **액세서리:** 장갑과 목도리를 추가하여 보온성을 높이는 것이 좋습니다. 특히 바람이 불 경우 체감온도가 더      │
│  낮아질 수 있으므로, 방풍 기능이 있는 액세서리를 고려하세요.                                                    │
│                                                                                                                 │
│  ## 최종 점검 사항                                 

# 2026년 2월 서울 대학교 졸업식 코디 추천

## 추천 코디 1안
- **상의:** 베이지 터틀넥
- **하의:** 브라운 슬랙스
- **신발:** 검은색 로퍼

### 품목별 선택 이유
- **베이지 터틀넥:** 따뜻하면서도 세련된 느낌을 주며, 졸업식의 격식 있는 분위기와 잘 어울립니다. 또한, 베이지 색상은 다양한 하의와 매치하기 용이합니다.
- **브라운 슬랙스:** 클래식한 슬랙스는 격식을 갖춘 스타일을 연출하며, 브라운 색상은 따뜻한 느낌을 더해줍니다. 슬랙스는 편안하면서도 세련된 실루엣을 제공합니다.
- **검은색 로퍼:** 졸업식에 적합한 포멀한 신발로, 편안하면서도 스타일리시한 선택입니다. 검은색은 어떤 색상과도 잘 어울립니다.

## 대체 코디 1안
- **상의:** 네이비 긴팔 티셔츠
- **하의:** 회색 치노팬츠
- **신발:** 회색 운동화

### 품목별 선택 이유
- **네이비 긴팔 티셔츠:** 기본 아이템으로, 격식 있는 자리에서도 무난하게 착용할 수 있습니다. 네이비 색상은 세련된 느낌을 주며, 다양한 하의와 잘 어울립니다.
- **회색 치노팬츠:** 편안하면서도 깔끔한 실루엣을 제공하여 졸업식에 적합합니다. 회색은 다양한 색상과 매치하기 좋습니다.
- **회색 운동화:** 편안함을 제공하면서도 스타일을 유지할 수 있는 신발입니다. 졸업식 후 외부에서의 활동에도 적합합니다.

## 날씨 대응 팁
- **겉옷:** 졸업식이 실내에서 진행되지만, 외부에서 대기할 가능성을 고려하여 따뜻한 겉옷을 준비하는 것이 좋습니다. 예를 들어, 네이비 트렌치코트나 회색 숏패딩을 추가로 착용하면 좋습니다.
- **액세서리:** 장갑과 목도리를 추가하여 보온성을 높이는 것이 좋습니다. 특히 바람이 불 경우 체감온도가 더 낮아질 수 있으므로, 방풍 기능이 있는 액세서리를 고려하세요.

## 최종 점검 사항
1. **날씨 확인:** 졸업식 당일의 날씨를 확인하여 기온과 바람 세기에 맞춰 복장을 조정하세요.
2. **편안함:** 졸업식 후 사진 촬영 등의 활동을 고려하여 편안한 신발을 선택하세요.
3. **레이어링:** 실내와 실외의 온도 차이를 고려하여 레이어링을 통해 체온을 조절할 수 있도록 준비하세요.

위의 추천 코디를 참고하여 따뜻하면서도 세련된 졸업식 복장을 준비하시기 바랍니다. 졸업식은 중요한 행사이므로, 자신감 있게 참석하시길 바랍니다!

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯